In [1]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

In [1]:
from ultralytics import YOLO

print("Ultralytics installed successfully!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/vaibhav_09/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics installed successfully!


### Download the pretrained model (YOLO)

In [2]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

### Load the YOLO Model

In [3]:
from ultralytics import YOLO

# Load pretrained YOLOv8 Nano model
yolo_model = YOLO("yolov8n.pt")

print("YOLO Model Loaded Successfully!")

YOLO Model Loaded Successfully!


In [5]:
for i, layer in enumerate(yolo_model.model.model):
    print(f"Layer {i}: {layer.__class__.__name__}")

Layer 0: Conv
Layer 1: Conv
Layer 2: C2f
Layer 3: Conv
Layer 4: C2f
Layer 5: Conv
Layer 6: C2f
Layer 7: Conv
Layer 8: C2f
Layer 9: SPPF
Layer 10: Upsample
Layer 11: Concat
Layer 12: C2f
Layer 13: Upsample
Layer 14: Concat
Layer 15: C2f
Layer 16: Conv
Layer 17: Concat
Layer 18: C2f
Layer 19: Conv
Layer 20: Concat
Layer 21: C2f
Layer 22: Detect


### Prepare for Feature Extraction

In [6]:
import torch
import torch.nn.functional as F
import cv2

# Dictionary to store intermediate features
feature_maps = {}

def hook_fn(module, input, output):
    feature_maps["layer21"] = output

# Register hook on Layer 21
hook = yolo_model.model.model[21].register_forward_hook(hook_fn)

# Dummy image
dummy = torch.randn(1, 3, 640, 640)

# Forward pass
with torch.no_grad():
    yolo_model.model(dummy)

# Print feature map shape
print("Layer 21 Feature Map Shape:", feature_maps["layer21"].shape)

# Remove hook
hook.remove()


Layer 21 Feature Map Shape: torch.Size([1, 256, 20, 20])


### Create the Embedding Extraction Function

In [7]:
import torch
import torch.nn.functional as F
import cv2
import numpy as np

# Store intermediate features
feature_maps = {}

def hook_fn(module, input, output):
    feature_maps["layer21"] = output

# Register hook once
hook = yolo_model.model.model[21].register_forward_hook(hook_fn)


def extract_yolo_embedding(image_path):
    """
    Extract a 256-dimensional semantic embedding using YOLOv8 Layer 21.
    """

    # Read image
    image = cv2.imread(image_path)

    if image is None:
        return np.zeros(256, dtype=np.float32)

    # YOLO expects RGB
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Forward pass
    with torch.no_grad():
        yolo_model.predict(image, verbose=False)

    # Feature map
    fmap = feature_maps["layer21"]

    # Global Average Pooling
    embedding = F.adaptive_avg_pool2d(fmap, (1, 1))

    # Flatten
    embedding = embedding.squeeze().cpu().numpy()

    return embedding

### Test the YOLO Embedding Function

In [8]:
# Select device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move YOLO model to device
yolo_model.model.to(DEVICE)
yolo_model.model.eval()

print(f"Using Device: {DEVICE}")

Using Device: cuda


In [ ]:
# Test on a single image

sample_image = "data/train/bright/0d84fdc2-fa66-4e46-bf69-e722562e38fd.png"   # <-- Change this path

embedding = extract_yolo_embedding(sample_image)

print("Embedding Shape :", embedding.shape)
print("Data Type       :", embedding.dtype)
print("First 10 Values :", embedding[:10])
print("Contains NaN    :", np.isnan(embedding).any())
print("Min Value       :", embedding.min())
print("Max Value       :", embedding.max())

Shape : (256,)
Dtype : float32
NaN   : False
Min   : 0.0
Max   : 0.0
First 10: [          0           0           0           0           0           0           0           0           0           0]


[ WARN:0@2988.591] global loadsave.cpp:278 findDecoder imread_('your_actual_image.jpg'): can't open/read file: check file path/integrity


### Improved extract_yolo_embedding()

In [11]:
import cv2
import numpy as np
import torch
import torch.nn.functional as F

# Dictionary to store intermediate features
feature_maps = {}

def hook_fn(module, input, output):
    feature_maps["layer21"] = output

# Register hook (run only once)
hook = yolo_model.model.model[21].register_forward_hook(hook_fn)


def extract_yolo_embedding(image_path):
    """
    Extract a 256-dimensional semantic embedding from YOLOv8 Layer 21.
    """

    # Read image
    image = cv2.imread(image_path)

    if image is None:
        return np.zeros(256, dtype=np.float32)

    # BGR -> RGB
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Resize to YOLO input size
    image = cv2.resize(image, (640, 640))

    # Normalize
    image = image.astype(np.float32) / 255.0

    # HWC -> CHW
    image = np.transpose(image, (2, 0, 1))

    # Tensor
    image = torch.from_numpy(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():

        # Direct forward pass
        _ = yolo_model.model(image)

        # Feature map from Layer 21
        fmap = feature_maps["layer21"]

        # Global Average Pooling
        embedding = F.adaptive_avg_pool2d(fmap, (1, 1))

        # Flatten
        embedding = embedding.squeeze().cpu().numpy()

    return embedding.astype(np.float32)

In [13]:
sample_image = "data/train/bright/0d84fdc2-fa66-4e46-bf69-e722562e38fd.png"   # <-- Change this path

embedding = extract_yolo_embedding(sample_image)

print("Shape :", embedding.shape)
print("Dtype :", embedding.dtype)
print("NaN   :", np.isnan(embedding).any())
print("Min   :", embedding.min())
print("Max   :", embedding.max())
print("First 10:", embedding[:10])

Shape : (256,)
Dtype : float32
NaN   : False
Min   : -0.25858667
Max   : 0.65953666
First 10: [  -0.084626     -0.2241    -0.24485    -0.17143    -0.19753   -0.032191    -0.16975    -0.16753    -0.19898   -0.046831]


### Extract Embeddings for the Entire Dataset

In [14]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm

# Dataset root
DATASET_PATH = Path("data/train")

records = []

# Class folders
classes = ["bright", "dark", "normal"]

for label_name in classes:

    class_path = DATASET_PATH / label_name

    for image_path in tqdm(sorted(class_path.glob("*")), desc=label_name):

        embedding = extract_yolo_embedding(str(image_path))

        row = {
            "id": image_path.name,
            "label": label_name
        }

        # 256 YOLO features
        for i, value in enumerate(embedding):
            row[f"yolo_{i}"] = float(value)

        records.append(row)

# Create DataFrame
yolo_df = pd.DataFrame(records)

print("Shape :", yolo_df.shape)

yolo_df.head()

normal: 100%|██████████| 500/500 [00:11<00:00, 43.84it/s]

Shape : (1500, 258)


,id,label,yolo_0,yolo_1,yolo_2,yolo_3,yolo_4,yolo_5,yolo_6,yolo_7,...,yolo_246,yolo_247,yolo_248,yolo_249,yolo_250,yolo_251,yolo_252,yolo_253,yolo_254,yolo_255
0,0033f6e8-2d10-4004-9d07-24d36ced6aef.png,bright,-0.110711,-0.209557,-0.255472,0.164696,-0.246690,0.086788,-0.195313,-0.061666,...,0.010145,-0.215879,-0.167574,-0.171692,-0.199733,0.444073,-0.239958,-0.190312,-0.130304,-0.253915
1,0052335d-b321-43d5-9141-df318f55363c.png,bright,-0.151134,-0.218542,-0.247199,-0.167266,-0.215641,-0.170193,-0.196788,-0.080473,...,-0.030727,-0.190818,-0.184931,-0.185665,-0.189937,0.427048,-0.203417,-0.200350,-0.093491,-0.211844
2,00f104f6-c965-4df1-9879-4b9e560240d1.png,bright,0.007431,-0.177073,-0.254069,-0.078289,-0.215449,-0.150797,-0.153334,-0.109329,...,-0.071400,-0.112720,-0.091539,-0.151469,-0.091600,0.526637,-0.178923,-0.186902,-0.135496,-0.208602
3,013d05de-6a66-4b4b-b65b-89e7eb854432.png,bright,0.056738,-0.189781,-0.214760,-0.105267,-0.195596,-0.041896,-0.144247,0.043115,...,-0.040816,-0.118011,-0.051263,-0.056770,-0.088346,0.536550,-0.202558,-0.151324,-0.132616,-0.213243
4,01960ea1-f99c-4e68-9fab-cab4ce8efabf.png,bright,-0.116196,-0.208650,-0.233707,-0.134883,-0.239375,-0.127405,-0.177594,-0.117084,...,-0.079801,-0.185786,-0.154499,-0.158038,-0.121723,0.432246,-0.228380,-0.225943,-0.166018,-0.212750


In [15]:
# Save the Features
yolo_df.to_csv("yolo_features.csv", index=False)

print("YOLO features saved successfully!")

YOLO features saved successfully!


In [16]:
df = pd.read_csv("yolo_features.csv")

print(df.shape)
print(df.head())

print("\nMissing Values :", df.isnull().sum().sum())
print("Duplicate IDs  :", df["id"].duplicated().sum())

(1500, 258)
                                         id   label    yolo_0    yolo_1  \
0  0033f6e8-2d10-4004-9d07-24d36ced6aef.png  bright -0.110711 -0.209557   
1  0052335d-b321-43d5-9141-df318f55363c.png  bright -0.151134 -0.218542   
2  00f104f6-c965-4df1-9879-4b9e560240d1.png  bright  0.007431 -0.177073   
3  013d05de-6a66-4b4b-b65b-89e7eb854432.png  bright  0.056738 -0.189781   
4  01960ea1-f99c-4e68-9fab-cab4ce8efabf.png  bright -0.116196 -0.208650   

     yolo_2    yolo_3    yolo_4    yolo_5    yolo_6    yolo_7  ...  yolo_246  \
0 -0.255472  0.164696 -0.246690  0.086788 -0.195313 -0.061666  ...  0.010145   
1 -0.247199 -0.167266 -0.215641 -0.170193 -0.196788 -0.080473  ... -0.030727   
2 -0.254069 -0.078289 -0.215449 -0.150797 -0.153334 -0.109329  ... -0.071400   
3 -0.214760 -0.105267 -0.195596 -0.041896 -0.144247  0.043115  ... -0.040816   
4 -0.233707 -0.134883 -0.239375 -0.127405 -0.177594 -0.117084  ... -0.079801   

   yolo_247  yolo_248  yolo_249  yolo_250  yolo_251  yol

### check on the YOLO features

In [17]:
import pandas as pd

yolo_df = pd.read_csv("yolo_features.csv")

feature_cols = [c for c in yolo_df.columns if c.startswith("yolo_")]

print("Feature Matrix Shape:", yolo_df[feature_cols].shape)

print("\nMinimum Standard Deviation:",
      yolo_df[feature_cols].std().min())

print("Maximum Standard Deviation:",
      yolo_df[feature_cols].std().max())

constant_features = (yolo_df[feature_cols].std() == 0).sum()

print("Constant Features:", constant_features)

Feature Matrix Shape: (1500, 256)

Minimum Standard Deviation: 0.010386045868782128
Maximum Standard Deviation: 0.19655862568824198
Constant Features: 0
